In [8]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape
import os

# ====== 경로 설정(사용시 변경) ======
tif_path = r"C:\Users\WJY\OneDrive\바탕 화면\SW\PNUSW-35\wjy\Data\산사태위험지도\47.tif"
clr_path = r"C:\Users\WJY\OneDrive\바탕 화면\SW\PNUSW-35\wjy\Data\산사태위험지도\47.clr"
png_path = r"C:\Users\WJY\OneDrive\바탕 화면\SW\PNUSW-35\wjy\Data\산사태위험지도\landslide_map.png"
shp_path = r"C:\Users\WJY\OneDrive\바탕 화면\SW\PNUSW-35\wjy\Data\산사태위험지도\landslide_polygons.shp"

# ====== .clr 파일 로드 함수 ======
def load_clr(clr_path):
    """CLR 파일에서 색상 정보를 로드"""
    value_to_rgb = {}
    try:
        with open(clr_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                parts = line.split()
                if len(parts) >= 4:
                    try:
                        value = int(parts[0])
                        rgb = tuple(int(p) for p in parts[1:4])
                        value_to_rgb[value] = rgb
                    except ValueError:
                        continue
        print(f"CLR 파일에서 {len(value_to_rgb)}개의 색상 정보를 로드했습니다.")
        return value_to_rgb
    except FileNotFoundError:
        print(f"CLR 파일을 찾을 수 없습니다: {clr_path}")
        return {}

# ====== Colormap 생성 ======
def make_colormap(value_to_rgb):
    """값-RGB 딕셔너리에서 matplotlib colormap 생성"""
    if not value_to_rgb:
        return plt.cm.viridis  # 기본 colormap

    # 값들을 정렬하여 연속적인 colormap 생성
    sorted_values = sorted(value_to_rgb.keys())
    colors = []

    for val in sorted_values:
        rgb = value_to_rgb[val]
        colors.append([c / 255.0 for c in rgb])

    return mcolors.ListedColormap(colors, name="hazard_cmap")

# ====== 시각화 + 저장 함수 ======
def visualize_and_export(tif_path, clr_path, png_path, shp_path, downsample_ratio=6):
    """TIF 파일을 시각화하고 PNG와 SHP 파일로 저장"""

    # 파일 존재 확인
    if not os.path.exists(tif_path):
        print(f"TIF 파일을 찾을 수 없습니다: {tif_path}")
        return False

    # CLR 파일 로드
    clr_dict = load_clr(clr_path)
    cmap = make_colormap(clr_dict)

    try:
        with rasterio.open(tif_path) as src:
            print(f"원본 이미지 크기: {src.width} x {src.height}")
            print(f"CRS: {src.crs}")
            print(f"NoData 값: {src.nodata}")

            # 다운샘플링
            out_shape = (1, src.height // downsample_ratio, src.width // downsample_ratio)
            transform = src.transform * src.transform.scale(
                src.width / out_shape[2],
                src.height / out_shape[1]
            )

            data = src.read(
                1,
                out_shape=out_shape,
                resampling=rasterio.enums.Resampling.mode
            )

            nodata = src.nodata if src.nodata is not None else 0
            original_crs = src.crs

            print(f"다운샘플링된 크기: {data.shape}")
            print(f"데이터 범위: {np.min(data)} ~ {np.max(data)}")

        # PNG 저장
        print("PNG 파일 생성 중...")
        data_masked = np.where(data == nodata, np.nan, data)

        plt.figure(figsize=(12, 10))

        # 데이터 값 범위 확인
        valid_data = data_masked[~np.isnan(data_masked)]
        if len(valid_data) > 0:
            vmin, vmax = np.min(valid_data), np.max(valid_data)
            print(f"유효한 데이터 범위: {vmin} ~ {vmax}")
        else:
            vmin, vmax = 1, 5

        img = plt.imshow(data_masked, cmap=cmap, vmin=vmin, vmax=vmax)

        # 컬러바 설정
        if len(clr_dict) > 0:
            sorted_values = sorted(clr_dict.keys())
            cbar = plt.colorbar(img, ticks=sorted_values)
            if len(sorted_values) == 5:  # 5단계 위험도인 경우
                cbar.ax.set_yticklabels([f"{v} {'(high)' if v==1 else '(low)' if v==5 else ''}"
                                       for v in sorted_values])
        else:
            plt.colorbar(img)

        plt.title("Landslide Susceptibility Map of Gyeongsangbuk-do", fontsize=14)
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"PNG 파일이 저장되었습니다: {png_path}")

        # SHP 저장
        print("SHP 파일 생성 중...")
        mask = data != nodata

        # 메모리 효율성을 위해 청크 단위로 처리
        print("폴리곤 생성 중... (시간이 걸릴 수 있습니다)")
        results = []

        for geom, val in shapes(data.astype(np.int32), mask=mask, transform=transform):
            results.append({
                "geometry": shape(geom),
                "properties": {
                    "value": int(val),
                    "risk_level": get_risk_level(int(val))
                }
            })

        print(f"총 {len(results)}개의 폴리곤이 생성되었습니다.")

        # CRS 설정 (원본 파일의 CRS 사용, 없으면 기본값)
        target_crs = original_crs if original_crs else "EPSG:5181"

        gdf = gpd.GeoDataFrame.from_features(results, crs=target_crs)
        gdf.to_file(shp_path, encoding='utf-8')
        print(f"SHP 파일이 저장되었습니다: {shp_path}")

        return True

    except Exception as e:
        print(f"오류 발생: {e}")
        return False

def get_risk_level(value):
    """위험도 값을 텍스트로 변환"""
    risk_levels = {
        1: "Very High",
        2: "High",
        3: "Moderate",
        4: "Low",
        5: "Very Low"
    }
    return risk_levels.get(value, f"Level_{value}")

# ====== 실행 ======
if __name__ == "__main__":
    print("산사태 위험도 맵 생성을 시작합니다...")
    success = visualize_and_export(tif_path, clr_path, png_path, shp_path, downsample_ratio=6)

    if success:
        print("\n✅ 모든 파일이 성공적으로 생성되었습니다!")
        print(f"PNG 파일: {png_path}")
        print(f"SHP 파일: {shp_path}")
    else:
        print("\n❌ 파일 생성 중 오류가 발생했습니다.")

ModuleNotFoundError: No module named 'rasterio'

In [3]:
!pip install rasterio matplotlib geopandas
!pip install rasterio
!pip install rioxarray

Defaulting to user installation because normal site-packages is not writeable
     --------------------------------------- 25.4/25.4 MB 11.3 MB/s eta 0:00:00
     ------------------------------------- 323.6/323.6 KB 10.1 MB/s eta 0:00:00
     -------------------------------------- 157.7/157.7 KB 9.2 MB/s eta 0:00:00
     ---------------------------------------- 98.2/98.2 KB 5.5 MB/s eta 0:00:00
     ---------------------------------------- 1.4/1.4 MB 23.1 MB/s eta 0:00:00
     --------------------------------------- 19.2/19.2 MB 13.4 MB/s eta 0:00:00
     ---------------------------------------- 6.1/6.1 MB 13.4 MB/s eta 0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'C:\Program Files\Python39\python.exe -m pip install --upgrade pip' command.


Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the 'C:\Program Files\Python39\python.exe -m pip install --upgrade pip' command.


Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 53.7/53.7 KB 1.4 MB/s eta 0:00:00
     ---------------------------------------- 1.2/1.2 MB 3.6 MB/s eta 0:00:00


You should consider upgrading via the 'C:\Program Files\Python39\python.exe -m pip install --upgrade pip' command.
